In [ ]:
# pip install langchain langchain-community sentence-transformers langchain-text-splitters
# pip install faiss-cpu

In [18]:
import pandas as pd

df_final = pd.read_csv('cleaned_data.csv')
print(df_final.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   feedback_id          11 non-null     int64 
 1   product              11 non-null     object
 2   customer_feedback    11 non-null     object
 3   word_count           11 non-null     int64 
 4   processed_feedback   11 non-null     object
 5   normalized_feedback  11 non-null     object
dtypes: int64(2), object(4)
memory usage: 656.0+ bytes
None


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import pandas as pd
import faiss
import numpy as np

# An AI has a "memory limit" called a context window. 
# We cannot feed it a 100-page manual all at once. We use the RecursiveCharacterTextSplitter to slice our text.
# Chunk Size (100): Each piece is about 100 characters long.
# Overlap (20): We repeat the last 20 characters of "Chunk 1" at the start of "Chunk 2."
# CHUNKING
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
all_chunks = []
for text in df_final['normalized_feedback']:
    chunks = text_splitter.split_text(text)
    all_chunks.extend(chunks)

- Computers don't understand words; they understand numbers. We turn our text chunks into Embeddings (lists of numbers).
- SentenceTransformer: This model converts the meaning of a chunk into a vector. If two chunks have similar meanings, their numbers will be mathematically "close" to each other.


In [ ]:


#  VECTORIZATION (EMBEDDINGS)
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(all_chunks).astype('float32')

embeddings

array([[-0.00860414, -0.00032323, -0.02718083, ...,  0.0030512 ,
        -0.06941408,  0.03882384],
       [ 0.02851582, -0.02358066, -0.01870342, ..., -0.05987552,
        -0.05539382, -0.02059784],
       [-0.08188844,  0.02667873,  0.05776942, ..., -0.12793325,
        -0.02255527, -0.02383981],
       ...,
       [-0.09549296,  0.05798148,  0.03428265, ..., -0.00837877,
         0.00859248, -0.01537954],
       [-0.06921981, -0.02605922, -0.05649259, ...,  0.02014002,
        -0.017565  ,  0.01594152],
       [-0.05909396, -0.00363895, -0.00580488, ..., -0.03112251,
         0.06706885, -0.0584183 ]], shape=(11, 384), dtype=float32)

- FAISS (Facebook AI Similarity Search): This is our high-speed library for searching vectors.
- Index: We "add" our vectors to a FAISS index. Think of this as an ultra-fast digital library where the librarian knows exactly where every "meaning" is located.

In [ ]:
# FAISS VECTOR STORAGE
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

- This is where we combine the user's question with the data we found in our library. This is the "Augmentation" part of RAG.
- Retrieval: When a user asks a question, we convert that question into a vector and ask FAISS: "Which chunk is most similar to this?"
- The Prompt Template: We don't just send the question to the LLM. We wrap it in a "System Role" and provide the "Context" found by FAISS.
- Instructional Engineering: We tell the AI: "Answer ONLY using the context." This is a safety guardrail that prevents the AI from making things up (hallucinating).

In [ ]:

# PROMPT ENGINEERING (THE RAG SYSTEM)
def ask_credi_trust(query):
    # 1. Retrieval
    query_vec = model.encode([query]).astype('float32')
    D, I = index.search(query_vec, k=1) # Find the top 1 match
    context = all_chunks[I[0][0]]
    
    # 2. Prompt Construction (The "Engineering" part)
    prompt = f"""
    SYSTEM ROLE: You are a professional CrediTrust Financial Analyst.
    
    CONTEXT (Evidence from database): {context}
    
    USER QUESTION: {query}
    
    INSTRUCTION: 
    - Answer the question using ONLY the context provided.
    - If the answer is not in the context, say "I cannot find this information in our records."
    - Tone: Professional and empathetic.
    """
    return prompt

In [31]:
# --- TEST THE FULL SYSTEM ---
test_query = "What is the issue with credit card?"
final_prompt = ask_credi_trust(test_query)

print("--- FINAL CLEANED DATA (Sample) ---")
print(df_final[['product', 'normalized_feedback']].head(3))
print("\n--- FAISS SEARCH RESULT ---")
print(f"Total Chunks in DB: {index.ntotal}")
print("\n--- GENERATED PROMPT READY FOR LLM ---")
print(final_prompt)

--- FINAL CLEANED DATA (Sample) ---
         product                                normalized_feedback
0    Credit Card  hate high interest rat credit card much call v...
1  Personal Loan                   loan application delay frustrate
2           BNPL                     buy pay later option gr8 issue

--- FAISS SEARCH RESULT ---
Total Chunks in DB: 11

--- GENERATED PROMPT READY FOR LLM ---

    SYSTEM ROLE: You are a professional CrediTrust Financial Analyst.
    
    CONTEXT (Evidence from database): bad service credit card really disappoint call
    
    USER QUESTION: What is the issue with credit card?
    
    INSTRUCTION: 
    - Answer the question using ONLY the context provided.
    - If the answer is not in the context, say "I cannot find this information in our records."
    - Tone: Professional and empathetic.
    
